# Assignment 6 (demo). Beating baselines in a competition

**mlcourse.ai – Open Machine Learning Course**

Task source: [mlcourse.ai](https://mlcourse.ai/book/topic06/demo_assignment06.html), author Yury Kashnitsky, CC BY-NC-SA 4.0.

Following simple baselines in the ["Alice" competition](https://www.kaggle.com/c/catch-me-if-you-can-intruder-detection-through-webpage-session-tracking2), go on with feature engineering and try to achieve ~ 0.955 (or higher) ROC AUC on the Public Leaderboard.

Data: download `train_sessions.csv`, `test_sessions.csv` and `site_dic.pkl` from the competition's Data tab and put them into a `data/` folder next to this notebook.

In [ ]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA_PATH = Path("data")
SEED = 17

## Reading data

In [ ]:
times = [f"time{i}" for i in range(1, 11)]
sites = [f"site{i}" for i in range(1, 11)]

train_df = pd.read_csv(DATA_PATH / "train_sessions.csv", index_col="session_id", parse_dates=times)
test_df = pd.read_csv(DATA_PATH / "test_sessions.csv", index_col="session_id", parse_dates=times)
train_df = train_df.sort_values(by="time1")

with open(DATA_PATH / "site_dic.pkl", "rb") as f:
    site_dict = pickle.load(f)
id_to_site = {v: k for k, v in site_dict.items()}

y_train = train_df["target"].values
train_df.shape, test_df.shape, y_train.mean()

## Sites as text: TF-IDF over site n-grams

In [ ]:
def sessions_to_text(df):
    return df[sites].fillna(0).astype(int).apply(
        lambda row: " ".join(id_to_site.get(s, "unknown").replace(".", "_") for s in row if s > 0), axis=1
    )


train_text = sessions_to_text(train_df)
test_text = sessions_to_text(test_df)

tfidf = TfidfVectorizer(ngram_range=(1, 5), max_features=100000, sublinear_tf=True, token_pattern=r"\S+")
X_train_sites = tfidf.fit_transform(train_text)
X_test_sites = tfidf.transform(test_text)
X_train_sites.shape

## Time features

In [ ]:
def time_features(df):
    start = df["time1"]
    end = df[times].max(axis=1)
    feats = pd.DataFrame(index=df.index)
    feats["hour"] = start.dt.hour
    feats["weekday"] = start.dt.weekday
    feats["month"] = start.dt.month
    feats["morning"] = (start.dt.hour <= 11).astype(int)
    feats["day"] = start.dt.hour.between(12, 18).astype(int)
    feats["evening"] = (start.dt.hour >= 19).astype(int)
    feats["year_month"] = start.dt.year * 100 + start.dt.month
    feats["duration"] = np.log1p((end - start).dt.total_seconds())
    feats["n_sites"] = df[sites].notna().sum(axis=1)
    feats["n_unique"] = df[sites].nunique(axis=1)
    return feats


train_time = time_features(train_df)
test_time = time_features(test_df)

ohe = OneHotEncoder(handle_unknown="ignore")
cat_cols = ["hour", "weekday", "month"]
X_train_cat = ohe.fit_transform(train_time[cat_cols])
X_test_cat = ohe.transform(test_time[cat_cols])

scaler = StandardScaler()
num_cols = ["year_month", "duration", "n_sites", "n_unique"]
X_train_num = scaler.fit_transform(train_time[num_cols])
X_test_num = scaler.transform(test_time[num_cols])

X_train = hstack([X_train_sites, X_train_cat, csr_matrix(X_train_num), csr_matrix(train_time[["morning", "day", "evening"]].values)]).tocsr()
X_test = hstack([X_test_sites, X_test_cat, csr_matrix(X_test_num), csr_matrix(test_time[["morning", "day", "evening"]].values)]).tocsr()
X_train.shape, X_test.shape

## Time-aware cross-validation

In [ ]:
time_split = TimeSeriesSplit(n_splits=10)

for C in [1, 3, 5, 10]:
    logit = LogisticRegression(C=C, random_state=SEED, solver="liblinear")
    scores = cross_val_score(logit, X_train, y_train, cv=time_split, scoring="roc_auc", n_jobs=-1)
    print(f"C={C}: mean ROC AUC = {scores.mean():.4f} ± {scores.std():.4f}")

## Training on all data and writing the submission

In [ ]:
best_C = 5
logit = LogisticRegression(C=best_C, random_state=SEED, solver="liblinear")
logit.fit(X_train, y_train)
test_pred = logit.predict_proba(X_test)[:, 1]

pd.DataFrame({"target": test_pred}, index=test_df.index).to_csv("submission_alice.csv", index_label="session_id")
pd.read_csv("submission_alice.csv").head()